## **Environment Setup**
Import required libraries and configure paths

In [ ]:
# Standard library imports
import sys
import os
import gc

# Third-party imports
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Configure path to custom modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

print("✅ Environment setup complete")
print(f"📂 Working directory: {os.getcwd()}")

## **Dataset Review (UCI Predictive Maintenance)**
Overview of the UCI Predictive Maintenance dataset: structure, features, failure modes, and target distribution.

**Dataset Information:**
- **Source**: UCI Machine Learning Repository
- **Task**: Binary classification (Healthy vs Anomaly)
- **Samples**: 10,000 observations
- **Features**: 5 sensor measurements
- **Target**: Machine failure indicator (0 = Healthy, 1 = Failure)

**Features:**
1. Air temperature [K]
2. Process temperature [K]
3. Rotational speed [rpm]
4. Torque [Nm]
5. Tool wear [min]

**Failure Modes:**
- Tool Wear Failure (TWF)
- Heat Dissipation Failure (HDF)
- Power Failure (PWF)
- Overstrain Failure (OSF)
- Random Failures

The dataset represents realistic industrial sensor data with class imbalance (more healthy samples than failures), making it ideal for anomaly detection in predictive maintenance.

In [ ]:
# Load UCI Predictive Maintenance Dataset
import pandas as pd

# Paths
base_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
data_path = os.path.join(base_dir, "dataset.csv")

# Check if data exists
if not os.path.exists(data_path):
    print(f"⚠️  Dataset not found at: {data_path}")
    print(f"📥 Please download the UCI Predictive Maintenance dataset")
    print(f"   Expected path: {data_path}")
else:
    # Load data
    df = pd.read_csv(data_path)
    
    print("📥 Loaded UCI Predictive Maintenance Dataset")
    print(f"   Shape: {df.shape} (rows x features)")
    print(f"\n📊 Dataset overview:")
    print(df.head())
    
    # Basic statistics
    print(f"\n📈 Dataset statistics:")
    print(f"   Total samples: {len(df):,}")
    print(f"   Features: {df.shape[1]}")
    
    # Check for target column
    if 'Target' in df.columns or 'Machine failure' in df.columns:
        target_col = 'Target' if 'Target' in df.columns else 'Machine failure'
        class_dist = df[target_col].value_counts()
        print(f"\n⚖️  Class distribution:")
        print(f"   Healthy (0): {class_dist.get(0, 0):,} ({class_dist.get(0, 0)/len(df)*100:.1f}%)")
        print(f"   Failure (1): {class_dist.get(1, 0):,} ({class_dist.get(1, 0)/len(df)*100:.1f}%)")
        print(f"   Imbalance ratio: {class_dist.get(0, 0)/class_dist.get(1, 0):.1f}:1")
    
    # Feature ranges
    print(f"\n📏 Feature ranges:")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols[:5]:  # Show first 5 numeric features
        print(f"   {col}: [{df[col].min():.2f}, {df[col].max():.2f}]")
    
    # Missing values check
    missing = df.isnull().sum().sum()
    print(f"\n🧹 Missing values: {missing}")

# **Preprocessing**

## **1. Feature Selection**

Select the 5 key sensor measurements for anomaly detection:
- Air temperature [K]
- Process temperature [K]  
- Rotational speed [rpm]
- Torque [Nm]
- Tool wear [min]

These features represent the physical measurements from the industrial equipment and are directly related to machine health status.

In [ ]:
# Cell: Feature Selection
print("📡 Feature Selection")
print("="*60)

# Define expected features (UCI dataset standard columns)
EXPECTED_FEATURES = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

# Check which features are available
available_features = [f for f in EXPECTED_FEATURES if f in df.columns]
missing_features = [f for f in EXPECTED_FEATURES if f not in df.columns]

print(f"✅ Feature selection complete:")
print(f"   Expected features: {len(EXPECTED_FEATURES)}")
print(f"   Available features: {len(available_features)}")

if missing_features:
    print(f"   ⚠️  Missing features: {missing_features}")
    print(f"   Using available features only")
    SELECTED_FEATURES = available_features
else:
    SELECTED_FEATURES = EXPECTED_FEATURES
    print(f"   ✅ All expected features found")

print(f"\n🎯 Selected features: {SELECTED_FEATURES}")

# Identify target column
target_col = None
for possible_target in ['Target', 'Machine failure', 'Failure']:
    if possible_target in df.columns:
        target_col = possible_target
        break

if target_col:
    print(f"🎯 Target column: '{target_col}'")
else:
    print("⚠️  Warning: Target column not found automatically")

## **2. Feature Extraction**
Extract selected sensor features and target labels from the dataset

In [ ]:
# Cell: Feature Extraction
print("🎯 Feature Extraction")
print("="*60)

# Extract feature matrix
X = df[SELECTED_FEATURES].copy()
y = df[target_col].copy() if target_col else None

print(f"📊 Feature matrix created:")
print(f"   Features shape: {X.shape}")
if y is not None:
    print(f"   Target shape: {y.shape}")
    print(f"   Target dtype: {y.dtype}")

# Verify data integrity
print(f"\n🔍 Data integrity check:")
print(f"   Feature NaN values: {X.isna().sum().sum()}")
if y is not None:
    print(f"   Target NaN values: {y.isna().sum()}")

# Handle missing values if any
if X.isna().sum().sum() > 0:
    print(f"\n🧹 Handling missing values...")
    X = X.fillna(X.mean())
    print(f"   ✅ Missing values filled with feature means")

if y is not None and y.isna().sum() > 0:
    print(f"   Removing samples with missing targets...")
    valid_mask = ~y.isna()
    X = X[valid_mask]
    y = y[valid_mask]
    print(f"   ✅ Cleaned dataset: {X.shape}")

print(f"✅ Feature extraction complete!")

## **3. Standardization (Z-Score Normalization)**

Apply standard scaling to normalize features to zero mean and unit variance.

**Formula**: $z = \frac{x - \mu}{\sigma}$

Where:
- $x$ = original feature value
- $\mu$ = feature mean
- $\sigma$ = feature standard deviation

This normalization approach is standard for neural network training and ensures all features contribute equally to the model.

In [ ]:
# Cell: Standardization
print("📏 Z-Score Normalization (Standardization)")
print("="*60)

print("📋 Applying standardization:")
print("   Formula: z = (x - μ) / σ")
print("   Result: Zero mean, unit variance")

# Calculate statistics before normalization
print(f"\n📊 Before normalization:")
print(f"   Feature means: {X.mean().values}")
print(f"   Feature stds: {X.std().values}")
print(f"   Value range: [{X.min().min():.2f}, {X.max().max():.2f}]")

# Apply standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=SELECTED_FEATURES)

# Verify normalization
print(f"\n✅ After normalization:")
print(f"   Feature means: {X_scaled.mean().values.round(6)}")
print(f"   Feature stds: {X_scaled.std().values.round(6)}")
print(f"   Value range: [{X_scaled.min().min():.2f}, {X_scaled.max().max():.2f}]")

print(f"\n✅ Standardization complete!")

## **4. Train-Test Split**

Split the dataset into training and testing sets following standard machine learning practices.

**Configuration:**
- Test size: 20% (2,000 samples)
- Train size: 80% (8,000 samples)
- Random state: 42 (for reproducibility)
- Stratified split: Maintains class distribution in both sets

In [ ]:
# Cell: Train-Test Split
print("✂️  Train-Test Split")
print("="*60)

# Configuration
TEST_SIZE = 0.2
RANDOM_STATE = 42

print(f"📋 Split configuration:")
print(f"   Test size: {TEST_SIZE*100:.0f}%")
print(f"   Train size: {(1-TEST_SIZE)*100:.0f}%")
print(f"   Random state: {RANDOM_STATE}")
print(f"   Stratification: Enabled (maintains class distribution)")

# Perform stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, 
    y, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_STATE,
    stratify=y if y is not None else None
)

print(f"\n📊 Split results:")
print(f"   Training samples: {len(X_train):,} ({len(X_train)/len(X_scaled)*100:.1f}%)")
print(f"   Test samples: {len(X_test):,} ({len(X_test)/len(X_scaled)*100:.1f}%)")

if y is not None:
    train_class_dist = pd.Series(y_train).value_counts()
    test_class_dist = pd.Series(y_test).value_counts()
    
    print(f"\n⚖️  Training set distribution:")
    print(f"   Healthy (0): {train_class_dist.get(0, 0):,} ({train_class_dist.get(0, 0)/len(y_train)*100:.1f}%)")
    print(f"   Failure (1): {train_class_dist.get(1, 0):,} ({train_class_dist.get(1, 0)/len(y_train)*100:.1f}%)")
    
    print(f"\n⚖️  Test set distribution:")
    print(f"   Healthy (0): {test_class_dist.get(0, 0):,} ({test_class_dist.get(0, 0)/len(y_test)*100:.1f}%)")
    print(f"   Failure (1): {test_class_dist.get(1, 0):,} ({test_class_dist.get(1, 0)/len(y_test)*100:.1f}%)")

print(f"\n✅ Train-test split complete!")

## **5. Reshape for CNN Input**

Reshape data for 1D CNN architecture: (samples, timesteps, features)

**CNN Input Requirements:**
- Input shape: (samples, 5, 1)
- 5 timesteps = 5 sensor features treated as time sequence
- 1 channel = single measurement stream

This reshaping treats the 5 sensor measurements as a temporal sequence, allowing the CNN to learn spatial patterns across different sensor readings.

In [ ]:
# Cell: CNN Reshaping
print("🪟 Reshape for CNN Input")
print("="*60)

print(f"📋 Current shape:")
print(f"   Train: {X_train.shape}")
print(f"   Test: {X_test.shape}")

# Reshape to 3D for CNN: (samples, features, channels)
# Treating each feature as a "timestep" in the sequence
X_train_cnn = X_train.values.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn = X_test.values.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"\n🎯 CNN input shape:")
print(f"   Train: {X_train_cnn.shape} (samples, timesteps, channels)")
print(f"   Test: {X_test_cnn.shape} (samples, timesteps, channels)")
print(f"\n   Interpretation:")
print(f"   • {X_train_cnn.shape[0]:,} training samples")
print(f"   • {X_train_cnn.shape[1]} timesteps (sensor features)")
print(f"   • {X_train_cnn.shape[2]} channel (single measurement stream)")

print(f"\n✅ Reshaping complete!")
print(f"   Ready for Conv1D layers")

## **6. Data Storage**
Save preprocessed data and configuration for model training

In [ ]:
# Cell: Save Preprocessed Data
print("💾 Saving Preprocessed Data")
print("="*60)

# Create results directory
results_dir = '../results/'
os.makedirs(results_dir, exist_ok=True)

# Save preprocessed arrays
np.save(results_dir + 'X_train.npy', X_train_cnn)
np.save(results_dir + 'X_test.npy', X_test_cnn)
np.save(results_dir + 'y_train.npy', y_train.values if y_train is not None else None)
np.save(results_dir + 'y_test.npy', y_test.values if y_test is not None else None)

# Save scaler for inference
import joblib
joblib.dump(scaler, results_dir + 'scaler.pkl')

# Save preprocessing configuration
import json
preprocessing_config = {
    'dataset': 'UCI Predictive Maintenance',
    'features': SELECTED_FEATURES,
    'n_features': len(SELECTED_FEATURES),
    'target_column': target_col,
    'train_samples': int(X_train_cnn.shape[0]),
    'test_samples': int(X_test_cnn.shape[0]),
    'input_shape': list(X_train_cnn.shape[1:]),
    'test_size': TEST_SIZE,
    'random_state': RANDOM_STATE,
    'normalization': 'StandardScaler (z-score)',
    'class_distribution': {
        'train': {
            'healthy': int(train_class_dist.get(0, 0)),
            'failure': int(train_class_dist.get(1, 0))
        },
        'test': {
            'healthy': int(test_class_dist.get(0, 0)),
            'failure': int(test_class_dist.get(1, 0))
        }
    } if y is not None else None
}

with open(results_dir + 'preprocessing_config.json', 'w') as f:
    json.dump(preprocessing_config, f, indent=2)

print(f"✅ Data saved successfully:")
print(f"   📂 Location: {os.path.abspath(results_dir)}")
print(f"   📊 Arrays: X_train.npy, X_test.npy, y_train.npy, y_test.npy")
print(f"   📏 Scaler: scaler.pkl")
print(f"   ⚙️  Config: preprocessing_config.json")

print(f"\n📦 Saved files:")
for filename in ['X_train.npy', 'X_test.npy', 'y_train.npy', 'y_test.npy', 'scaler.pkl', 'preprocessing_config.json']:
    filepath = os.path.join(results_dir, filename)
    if os.path.exists(filepath):
        size = os.path.getsize(filepath) / 1024  # KB
        print(f"   ✅ {filename} ({size:.1f} KB)")

## **7. Preprocessing Summary**
Complete overview of the preprocessing pipeline implementation

In [ ]:
# Cell: Final Summary
print("🎯 PREPROCESSING COMPLETE - Detection Pipeline")
print("="*70)

print("📋 COMPLETED PREPROCESSING STEPS:")
print("   ✅ 1. Feature Selection: 5 sensor measurements")
print("   ✅ 2. Feature Extraction: Separate features and target")
print("   ✅ 3. Standardization: Z-score normalization (μ=0, σ=1)")
print("   ✅ 4. Train-Test Split: 80/20 stratified split")
print("   ✅ 5. CNN Reshaping: 3D tensor for Conv1D layers")
print("   ✅ 6. Data Storage: Saved to ../results/ directory")

print(f"\n📊 FINAL DATASET STATISTICS:")
print(f"   🚂 Training: {X_train_cnn.shape[0]:,} samples")
print(f"   🧪 Testing:  {X_test_cnn.shape[0]:,} samples")
print(f"   📏 Input Shape: {X_train_cnn.shape[1:]} (timesteps, channels)")
print(f"   🎯 Features: {len(SELECTED_FEATURES)}")

if y is not None:
    imbalance_ratio = train_class_dist.get(0, 0) / train_class_dist.get(1, 0)
    print(f"\n⚖️  CLASS IMBALANCE:")
    print(f"   Ratio: {imbalance_ratio:.1f}:1 (Healthy:Failure)")
    print(f"   Recommendation: Use SMOTE or class weights during training")

print(f"\n💾 SAVED OUTPUTS:")
print(f"   📂 Directory: ../results/")
print(f"   🗂  Arrays: X_train.npy, X_test.npy, y_train.npy, y_test.npy")
print(f"   📏 Scaler: scaler.pkl (for inference preprocessing)")
print(f"   ⚙️  Config: preprocessing_config.json")

print(f"\n🚀 NEXT STEPS:")
print(f"   1. 🏗  Build CNN-RNN model architecture")
print(f"   2. ⚖️  Apply SMOTE or class weights for imbalance")
print(f"   3. 🎓 Train model with early stopping")
print(f"   4. 📈 Evaluate with recall, precision, F1-score")
print(f"   5. 🎯 Optimize for high recall (catch failures)")

print(f"\n✨ CONCLUSION:")
print(f"   The UCI Predictive Maintenance dataset has been successfully")
print(f"   preprocessed for anomaly detection. Data is normalized, split,")
print(f"   and reshaped for CNN training. The preprocessing pipeline is")
print(f"   ready for model development with proper handling of class")
print(f"   imbalance and feature scaling.")

print(f"\n🎯 Ready for CNN model training!")